In [ ]:
# ADJUST THIS CODE FROM PROCESS_FMCW_OUTPUT, MAKE INTO .PY SCRIPT WITH INPUTS

# split radagrams & coordinates horizontally
hmin_idx = 5000 # use same hmin_idx as above (to create square images)
hsplit_idxs = np.arange(0, all_rd.shape[1], hmin_idx)
hsplit_idxs = hsplit_idxs[1:] # grab split locations

Isurf_thresh = 800 # maximum depth of surface picks (in index number)
adjust_peaks = False # do peak adjustment
peak_shift = 70 # cm maximum peak shift to look for
velocity = 2.25e10

all_depths = []; all_depth_xs = []; all_depth_ys = []
# perform the splitting
counter = 1
for i in range(0, len(hsplit_idxs)-1): 
    # split radargrams and coordinates
    if i == len(hsplit_idxs)-2: # for the last split, include all the way to the end of the all_rd
        rd_split = all_rd[:,hsplit_idxs[i]:all_rd.shape[1]] 
        xs_split = xs_flattened[hsplit_idxs[i]:all_rd.shape[1]] # x coordinates
        ys_split = ys_flattened[hsplit_idxs[i]:all_rd.shape[1]] # y coordinates
    else: # for all others:
        rd_split = all_rd[:,hsplit_idxs[i]:hsplit_idxs[i+1]]
        xs_split = xs_flattened[hsplit_idxs[i]:hsplit_idxs[i+1]] # x coordinates
        ys_split = ys_flattened[hsplit_idxs[i]:hsplit_idxs[i+1]] # y coordinates

    # show the split radargram
    fig, ax1 = plt.subplots(1,figsize=(6,6))
    ax1.imshow(rd_split, vmin=np.percentile(rd_split,50), vmax=np.percentile(rd_split,100), cmap='Greys_r')
    ax1.set_title('preprocessed rd')
    plt.show()

    # WTMM
    radargram = rd_split; radargram[radargram < np.percentile(radargram,50)] = np.percentile(radargram,50)

    # export radaragram if it doesn't exist:
    if not os.path.exists('/Users/jukesliu/Documents/POSTDOC/snow-radar/ReynoldsMountain/ProcessedMay25_xyz/rd'+str(counter).zfill(2)+'_preprocessed.npy'):
        np.save('/Users/jukesliu/Documents/POSTDOC/snow-radar/ReynoldsMountain/ProcessedMay25_xyz/rd'+str(counter).zfill(2)+'_preprocessed.npy', radargram)
    else:
        print('rd saved already.')
        
     # wtmm_scale = 300
    # size_thresh = 100
    # mod_thresh_multiplier = 1.7
    for wtmm_scale in [1000, 800]:
        for size_thresh in [200,100]:
            for mod_thresh_multiplier in [1]:
                rd_depths = []; rd_depth_xs = []; rd_depth_ys = []; rd_surf_idxs = []; rd_ground_idxs = []; rd_x_idxs = []
                print('rd'+str(counter)+'_'+str(wtmm_scale)+'scale_'+str(size_thresh)+'size_'+str(mod_thresh_multiplier)+'mod')
                # if os.path.exists('/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/rd'+str(counter)+'_'+str(wtmm_scale)+'scale_'+str(size_thresh)+'size_'+str(mod_thresh_multiplier)+'mod.csv'):
                #     print('file exists. skipping to next:')
                # else:
                # if counter == 1:
                if counter >= 32: # start / stop at a certain rd
                    x_idxs = []
                    surf_idxs = []
                    ground_idxs = []
                    depths = [] 
                    
                    # Run the WTMM
                    # print(wtmm_scale)
                    [dx,dy,mm,m,a] = wtmm2d_v2(radargram,'gauss',wtmm_scale)  
                    
                    # # Visualize outputs from wtmm2d:
                    # fig, axs = plt.subplots(2,3,figsize=(15,10))
                    # axs[0,0].imshow(dx, aspect='equal', cmap = 'gray', interpolation='none'); axs[0,0].set_title('dx') # x gradient
                    # axs[0,1].imshow(dy, aspect='equal', cmap = 'gray', interpolation='none'); axs[0,1].set_title('dy') # y gradient
                    # axs[0,2].imshow(a, aspect='equal', cmap = 'gray', interpolation='none'); axs[0,2].set_title('a') # argument            
                    # axs[1,0].imshow(mm, aspect='equal', cmap = 'gray', interpolation='none', vmin = np.min(mm), vmax = np.max(m)); 
                    # axs[1,0].set_title('mm') # modulus maxima (interpolated)
                    # axs[1,1].imshow(m, aspect='equal', cmap = 'gray', interpolation='none',vmin = np.min(mm), vmax = np.max(m));
                    # axs[1,1].set_title('m') # modulus
                    # axs[-1,-1].axis('off')
                    # plt.show()
                    
                    # Chain the traces
                    cmm = wtmmchains(mm,a,0,wtmm_scale,wtmm_scale/4) # chain at a specified scale (UNMASKED)
                    
                    # Filter chains based on size threshold
                    cmm_passed = []
                    mods = []
                    for j in range(0, len(cmm)):
                        if cmm[j].size > size_thresh: # adjust this condition to threshold
                            cmm_passed.append(cmm[j])
                            mods.append(cmm[j].linemeanmod) # collect mod information
                    
                    # Filter again based on gradient values (mean modulus)
                    cmm_passed_2 = []
                    # Filter chains based on size and mod threshold
                    for k in range(0, len(cmm_passed)):
                           if cmm_passed[k].linemeanmod > np.nanmean(mods)*mod_thresh_multiplier:
                            cmm_passed_2.append(cmm_passed[k])
                    
                    # STRATEGY TO GRAB FIRST RETURNS FROM CHAINS
                    xs = []; ys = [] # grab all chain coordinates
                    for n in range(0, len(cmm_passed_2)):
                        xs.extend(cmm_passed_2[n].ix); ys.extend(cmm_passed_2[n].iy)
                    chain_coords = list(zip(xs,ys)); chain_coords.sort() # sort by xs
                    [xs,ys] = zip(*chain_coords)
                    xs = list(xs); ys = list(ys)
                
                    # PLOT RESULTS:
                    a=0.3
                    fig,ax = plt.subplots(1,figsize=(15,5))
                    ax.imshow(np.array(radargram),cmap='gray')
                    ax.set_xlim([0, np.array(radargram).shape[1]])
                    ax.set_ylim([0, np.array(radargram).shape[0]])
                    ax.invert_yaxis(); ax.set_aspect('equal')
                    
                    # FOR EACH COLUMN GRAB FIRST AND SECOND RETURN AND SUBTRACT (ONLY IF THERE ARE AT LEAST TWO), WINDOW REMOVED
                    for x_idx in range(0,radargram.shape[1]): 
                        coord_idxs = np.where(np.array(xs) == x_idx)
                        coord_idxs = coord_idxs[0]
                        # if there are at least 2 crossings at the x
                        if len(coord_idxs) > 2:
                            cxs = []; cys = []
                            for idx in coord_idxs:
                                cxs.append(xs[idx]); cys.append(ys[idx]) # grab the x and y coordinates
                    
                            # grab the first pair of ys that is > 1 apart
                            diff_idxs = np.where(np.diff(cys) > 1)[0]
                            if len(diff_idxs) > 0:
                                diff_first = diff_idxs[0]
                                diff_last = diff_idxs[-1]+1
                                Isurf = int(cys[diff_first]) # grab the returns
                                # Iground = int(cys[diff_first+1]) # grab the paired return
                                Iground = int(cys[diff_last]) # grab the last return

                                if adjust_peaks: # LOOK WITHIN 40CM OF IGROUND
                                    trace = radargram[:Iground,x_idx] # grab the trace up until the ground pick
                                    peak_idxs, _ = find_peaks(-trace)
                                    peak_idxs = peak_idxs[peak_idxs > len(trace)-round((peak_shift*2)/velocity/TWT_interval)] # keep only the peak idxs withinpeak shift
                                    max_peak_idx = peak_idxs[np.argmax(-trace[peak_idxs])]; Iground = max_peak_idx # grab the max peak
                                    # lowest_peak = np.max(peak_idxs); Iground = lowest_peak # grab the deepest index as new ground index
                                    
                                if Isurf <= Isurf_thresh and Iground >= Isurf_thresh: # look within upper portion for surface return, ground peak must be below
                                    depth = (TWT[Iground]-TWT[Isurf])*velocity/2 # calculate depth from TWTs
                                    # print(depth)
                                    ax.plot(x_idx, Isurf, 'b.', alpha=a)# plot the surface return
                                    ax.plot(x_idx, Iground, 'r.', alpha=a)# plot the ground return
                                    # depths.append(depth)
                                    
                                    all_depths.append(depth[0]); rd_depths.append(depth[0])
                                    all_depth_xs.append(xs_split[x_idx]); rd_depth_xs.append(xs_split[x_idx])
                                    all_depth_ys.append(ys_split[x_idx]); rd_depth_ys.append(ys_split[x_idx])
                                    rd_surf_idxs.append(Isurf); rd_ground_idxs.append(Iground)
                                    rd_x_idxs.append(x_idx)
                                else:
                                    all_depths.append(np.NaN); rd_depths.append(np.NaN)
                                    all_depth_xs.append(np.NaN); rd_depth_xs.append(np.NaN)
                                    all_depth_ys.append(np.NaN); rd_depth_ys.append(np.NaN)
                                    rd_surf_idxs.append(np.NaN); rd_ground_idxs.append(np.NaN)
                                    rd_x_idxs.append(np.NaN)
                            else:
                                # depths.append(np.NaN)
                                all_depths.append(np.NaN); rd_depths.append(np.NaN)
                                all_depth_xs.append(np.NaN); rd_depth_xs.append(np.NaN)
                                all_depth_ys.append(np.NaN); rd_depth_ys.append(np.NaN)
                                rd_surf_idxs.append(np.NaN); rd_ground_idxs.append(np.NaN)
                                rd_x_idxs.append(np.NaN)
                
                    # write to CSV
                    export_df = pd.DataFrame(list(zip(rd_depth_xs, rd_depth_ys, rd_depths, rd_surf_idxs, rd_ground_idxs, rd_x_idxs)), columns=['x','y','depth', 'Isurf','Iground','x_idx'])
                    export_df.to_csv('/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/rd'+str(counter).zfill(2)+'_'+str(wtmm_scale)+'scale_'+str(size_thresh)+'size_'+str(mod_thresh_multiplier)+'mod.csv')
                    plt.tight_layout() # display as figure
                    ax.legend(['surface','ground'])
                    plt.savefig('/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/rd'+str(counter).zfill(2)+'_'+str(wtmm_scale)+'scale_'+str(size_thresh)+'size_'+str(mod_thresh_multiplier)+'mod.jpg',dpi=200)
                    plt.show()
    counter+=1 # count the split radargrams